In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

try:
    # 1. Load the dataset
    train_df = pd.read_csv('data/train.tsv', sep='\t', header=None)
    test_df = pd.read_csv('data/test.tsv', sep='\t', header=None)

    # Extract the text (shifted to index 3) and labels (shifted to index 2)
    X_train = train_df[3].fillna('') 
    y_train = train_df[2].astype(str) 
    X_test = test_df[3].fillna('')
    y_test = test_df[2].astype(str)

    # Print a sample to confirm we are grabbing the right columns now
    print("Sample labels being used:", y_train.head(3).tolist())

    # Convert multi-class labels to binary (Fake vs Real) safely
    valid_labels = ['true', 'mostly-true', 'half-true']
    
    y_train_binary = y_train.apply(lambda x: 1 if x.lower().strip() in valid_labels else 0)
    y_test_binary = y_test.apply(lambda x: 1 if x.lower().strip() in valid_labels else 0)

    # 2. Vectorize the text using TF-IDF
    print("Vectorizing text...")
    vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    # 3. Train the Logistic Regression Model
    print("Training model...")
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_tfidf, y_train_binary)

    # 4. Evaluate the model
    predictions = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test_binary, predictions)
    print(f"\nSUCCESS! Model Test Accuracy: {accuracy * 100:.2f}%")

    # 5. Save the model and vectorizer for the Flask API
    print("Saving model and vectorizer...")
    joblib.dump(model, 'model.pkl')
    joblib.dump(vectorizer, 'vectorizer.pkl')
    print("Done! Files saved.")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

Sample labels being used: ['false', 'half-true', 'mostly-true']
Vectorizing text...
Training model...

SUCCESS! Model Test Accuracy: 61.56%
Saving model and vectorizer...
Done! Files saved.
